In [11]:
import simpy
import random

In [12]:
# Global variables
wait_table_times = []
wait_food_times = []
total_times = []
table_occupancy_times = []
chef_usage_times = []
waitstaff_usage_times = []
event_log = []


In [13]:
# This function records every event in the system. 

def log_event(env, name, event, restaurant, wait_val=None):
    event_log.append((
        env.now, name, event,
        len(restaurant.tables.queue),
        len(restaurant.chefs.queue),
        len(restaurant.waitstaff.queue),
        len(restaurant.tables.users),
        len(restaurant.chefs.users),
        len(restaurant.waitstaff.users),
        wait_val
    ))

In [14]:
# This class defines all the resources of the restaurant.

class Restaurant:
    def __init__(self, env, num_tables, num_chefs, num_wait, num_stations):
        self.tables = simpy.Resource(env, num_tables)
        self.chefs = simpy.Resource(env, num_chefs)
        self.waitstaff = simpy.Resource(env, num_wait)
        self.stations = simpy.Resource(env, num_stations)

In [15]:
# Generates customers according to an exponential inter-arrival process.

def customer_arrival(env, restaurant):
    customer_id = 0
    while True:
        yield env.timeout(random.expovariate(1/8))
        customer_id += 1
        env.process(customer(env, f"Customer {customer_id}", restaurant))

In [16]:
# This process represents the complete customer journey from arrival to exit.

def customer(env, name, restaurant):
    arrival = env.now
    log_event(env, name, "Arrived", restaurant)

    # Seating
    log_event(env, name, "Requested Table", restaurant)
    seat_req = env.now

    with restaurant.tables.request() as req:
        yield req
        wait_table = env.now - seat_req
        wait_table_times.append(wait_table)
        seat_time = env.now
        log_event(env, name, "Seated", restaurant, wait_table)
        yield env.timeout(random.uniform(1, 3))

    # Order Taking
    log_event(env, name, "Order Taking Requested", restaurant)
    with restaurant.waitstaff.request() as req:
        yield req
        ws_start = env.now
        log_event(env, name, "Order Taking Started", restaurant)
        yield env.timeout(random.uniform(2, 4))
        ws_end = env.now
        waitstaff_usage_times.append(ws_end - ws_start)
        log_event(env, name, "Order Taken", restaurant)
# Cooking
    log_event(env, name, "Cooking Requested", restaurant)
    cook_req = env.now

    with restaurant.chefs.request() as chef, restaurant.stations.request() as station:
        yield chef & station
        wait_food = env.now - cook_req
        wait_food_times.append(wait_food)
        chef_start = env.now
        log_event(env, name, "Cooking Started", restaurant, wait_food)
        yield env.timeout(abs(random.normalvariate(10, 2)))
        chef_end = env.now
        chef_usage_times.append(chef_end - chef_start)
        log_event(env, name, "Cooking Finished", restaurant)

    # Serving
    log_event(env, name, "Serving Requested", restaurant)
    with restaurant.waitstaff.request() as req:
        yield req
        ws_start = env.now
        log_event(env, name, "Serving Started", restaurant)
        yield env.timeout(random.uniform(1, 3))
        ws_end = env.now
        waitstaff_usage_times.append(ws_end - ws_start)
        log_event(env, name, "Served", restaurant)

    # Dining
    log_event(env, name, "Dining Started", restaurant)
    yield env.timeout(abs(random.normalvariate(30, 10)))
    log_event(env, name, "Dining Finished", restaurant)

    # Payment
    log_event(env, name, "Payment Requested", restaurant)
    with restaurant.waitstaff.request() as req:
        yield req
        ws_start = env.now
        log_event(env, name, "Payment Started", restaurant)
        yield env.timeout(random.uniform(2, 5))
        ws_end = env.now
        waitstaff_usage_times.append(ws_end - ws_start)
        log_event(env, name, "Payment Complete", restaurant)
 # Leaving
    total = env.now - arrival
    total_times.append(total)
    table_occ = env.now - seat_time
    table_occupancy_times.append(table_occ)
    log_event(env, name, "Left", restaurant, total)

In [17]:
# This function runs the entire simulation, resets lists, and computes statistics.

def run_sim(sim_time=600, seed=None):
    global wait_table_times, wait_food_times, total_times, table_occupancy_times
    global chef_usage_times, waitstaff_usage_times, event_log

    wait_table_times = []
    wait_food_times = []
    total_times = []
    table_occupancy_times = []
    chef_usage_times = []
    waitstaff_usage_times = []
    event_log = []

    if seed:
        random.seed(seed)

    env = simpy.Environment()
    restaurant = Restaurant(env, 20, 4, 6, 4)

    env.process(customer_arrival(env, restaurant))
    env.run(until=sim_time)

    def avg(lst):  
        return sum(lst) / len(lst) if len(lst) else 0

    result = (
        len(total_times),
        avg(wait_table_times),
        avg(wait_food_times),
        avg(total_times),
        avg(table_occupancy_times),
        sum(chef_usage_times) / (4 * sim_time),
        sum(waitstaff_usage_times) / (6 * sim_time)
    )

    return result

In [18]:
results = run_sim()

In [19]:
(total_customers,
 avg_wait_table,
 avg_wait_food,
 avg_total_time,
 avg_table_occ,
 chef_util,
 waitstaff_util) = results

print("\n--- Simulation Results ---")
print(f"Total Customers Served: {total_customers}")
print(f"Average Wait for Table: {avg_wait_table:.2f} min")
print(f"Average Wait for Food: {avg_wait_food:.2f} min")
print(f"Average Total Time: {avg_total_time:.2f} min")
print(f"Average Table Occupancy: {avg_table_occ:.2f} min")

print("\n--- Staff Utilization ---")
print(f"Chef Utilization: {chef_util:.3f} ({chef_util*100:.1f}%)")
print(f"Waitstaff Utilization: {waitstaff_util:.3f} ({waitstaff_util*100:.1f}%)")


--- Simulation Results ---
Total Customers Served: 47
Average Wait for Table: 0.00 min
Average Wait for Food: 0.00 min
Average Total Time: 50.98 min
Average Table Occupancy: 50.98 min

--- Staff Utilization ---
Chef Utilization: 0.198 (19.8%)
Waitstaff Utilization: 0.118 (11.8%)


In [20]:
print("\n--- Simulation Table (First 25 Events) ---")
header = f"{'Time':<7} {'Customer':<12} {'Event':<25} {'TblQ':<5} {'ChefQ':<5} {'WaitQ':<5} {'TblBusy':<8} {'ChefBusy':<8} {'WaitBusy':<9} {'Wait':<6}"
print(header)
print("-" * len(header))

for e in event_log[:25]:
    print(f"{e[0]:<7.2f} "
          f"{e[1]:<12} "
          f"{e[2]:<25} "
          f"{e[3]:<5} "
          f"{e[4]:<5} "
          f"{e[5]:<5} "
          f"{e[6]:<8} "
          f"{e[7]:<8} "
          f"{e[8]:<9} "
          f"{'-' if e[9] is None else f'{e[9]:.2f}':<6}")



--- Simulation Table (First 25 Events) ---
Time    Customer     Event                     TblQ  ChefQ WaitQ TblBusy  ChefBusy WaitBusy  Wait  
---------------------------------------------------------------------------------------------------
44.54   Customer 1   Arrived                   0     0     0     0        0        0         -     
44.54   Customer 1   Requested Table           0     0     0     0        0        0         -     
44.54   Customer 1   Seated                    0     0     0     1        0        0         0.00  
44.90   Customer 2   Arrived                   0     0     0     1        0        0         -     
44.90   Customer 2   Requested Table           0     0     0     1        0        0         -     
44.90   Customer 2   Seated                    0     0     0     2        0        0         0.00  
46.16   Customer 2   Order Taking Requested    0     0     0     1        0        0         -     
46.16   Customer 2   Order Taking Started      0     0  